# NHANES Liver Stiffness (LSM) — overview

Estimates the population distribution of liver stiffness measurement (LSM, kPa) in U.S. adults, for use in the consuming simulation project's Model 3 liver-stiffness exposure (which the trial intervention's MASLD/cirrhosis eligibility filter reads at enrollment).

## Data source

**NHANES 2017 – March 2020 pre-pandemic combined release (`P_LUX`).** This is the first NHANES cycle to include transient elastography (FibroScan); the COVID-19 pandemic forced an early stop, so NCHS combined the partial 2019–2020 fieldwork with the full 2017–2018 cycle into a single "pre-pandemic" public-use file. The combined release uses `WTMECPRP` as the MEC examination weight and the same `SDMVSTRA` × `SDMVPSU` paired-PSU variance design as the rest of continuous NHANES.

Variable of interest: `LUXSMED` (median liver stiffness, kPa). Auxiliary: `LUXSIQR` (IQR of valid measurements; QC criterion is `IQR / median < 0.30`), `LUAXSTAT` (examination completeness flag).

A second NHANES cycle (2021–2023, `LUX_L`) also collects FibroScan data; this analysis sticks with the pre-pandemic combined release for clean weight handling. Combining cycles is a follow-up step.

## Project context

The project's current `LIVER_STIFFNESS_*_KPA_STUB` (`mean = 6, SD = 4`) was set as a coarse all-adult literature placeholder (median ≈ 5 kPa, ~7 % above 9.5 kPa); the project's lognormal moment-matching makes that stub put roughly 7 % of simulants above the 12.5 kPa F4 cutoff. **Unlike Lp(a), LSM should vary substantially by age and sex** — fibrosis accumulates with age, and adult men carry higher MASLD prevalence than adult women — so a single scalar may be a worse approximation here.

## Notebooks

1. `01_download_lux.ipynb` — fetch `P_DEMO.xpt` and `P_LUX.xpt` from CDC (cached on disk); merge demographics and elastography into a tidy parquet at `data/derived/nhanes_p_lux.parquet`.
2. `02_lsm_marginal.ipynb` — survey-weighted age × sex marginal with 95 % CIs from a paired-PSU jackknife. Compare the all-adult and trial-band (65–80) mean and SD against the project's `(6, 4)` stub. Report between-cell vs within-cell SD ratio to decide whether single-scalar is adequate.
3. `03_lsm_transformations.ipynb` — robustness of the trial-band scalars under raw / sqrt / log transformations, plus outlier sensitivity and lognormal upper-tail goodness-of-fit. Mirrors the equivalent notebook in `nhanes_lpa_distribution/`.

## Caveats

- **F4 prevalence is the load-bearing quantity** — the project's stub is calibrated so that the *fraction* of simulants above 12.5 kPa matches the literature ~7 % F4 prevalence. The empirical NHANES F4 share by age × sex is the right cross-check, not just the mean and SD.
- **MEC-only sample.** FibroScan is a MEC procedure; weights are `WTMECPRP`. Respondents with `LUAXSTAT ≠ 1` (incomplete exam, refused, ineligible) are dropped.
- **Combined-cycle weights.** `WTMECPRP` is the proper sampling weight for the 2017 – March 2020 combined release. NHANES analytic guidance for combining `P_*` with later cycles (`LUX_L` 2021–2023) requires extra weight scaling; this notebook stays within the single combined release.